# T1b — 근거 다이어트 품질 재설계 (대조군 + 답마다 절대 평가)

## 왜 하나

T1(11/23)에서 근거 다이어트는 LLM 입력 토큰을 **−73%** 줄였지만 품질 기준(쌍 비교 WORSE ≤ 10%)에서 미달했다(37.4%).
그런데 다이어트가 거의 없던 S 조건도 WORSE 36%였다 — 같은 입력을 두 번 생성해도 판정자가 한쪽을 고르는 **바닥값**이었고, 답의 31~41%가 512토큰에서 잘렸다.
기준 설계의 결함이므로, 대조군을 넣고 답마다 따로 평가해 **"다이어트가 품질을 떨어뜨리는가"**를 다시 잰다.

## 무엇을 하나 (학습 없음, CPU)

- **데이터**: RAGTruth train QA 근거 중 **T1의 200개를 뺀** 풀에서 `Random(2038)` 200개. 근거는 T1의 L과 같은 방식(원래 지문 + 다른 근거 7개)
- **팔 셋**(`claude-haiku-5-5`, 배치, **temperature 0, max_tokens 1024**): `full`(전체 근거), `full2`(같은 입력 재생성 = 대조군), `trim`(게이트웨이 Trimmer top_k 4)
- **평가**(`claude-opus-5-5`, 배치, 답마다 따로·어느 팔인지 가림): ① 정답성 yes/partial/no(원래 지문 기준), ② 충실성(R4 문장 S/H/A, H 하나라도 → 환각 응답)

## 사전 정의 기준 (SSOT §6 T1b, 실행 전 고정)

- **T1b-0(유효성)**: 세 팔 모두 생성·평가 성공한 질문 ≥ 180, 그리고 **대조군 안정성** `yes` 비율 |full2 − full| ≤ 0.05 (넘으면 판정 보류)
- **T1b-1(절감)**: LLM 입력 토큰 중앙값 감소(trim vs full) ≥ 40%
- **T1b-2(품질 주, 비열등)**: `yes` 비율 trim − full ≥ −0.05, 그리고 쌍체 부트스트랩(2,000회, 시드 2038) 95% 하한 ≥ −0.10
- **T1b-3(충실성)**: 환각 응답 비율 trim − full ≤ +0.03, 95% 상한 < +0.08
- **채택 = T1b-0~3** → `trim`을 비용 절감 권장 옵션으로 문서화(선택 기능 유지)
- temperature 0이 거부되면 형식 확인에서 빼고 기본값으로 진행, 그 사실을 기록(사전 고정). **API 상한 $12**

> 런타임: **CPU**. Colab 비밀 값 `ANTHROPIC_API_KEY`. 끊기면 다시 모두 실행(드라이브 `t1b/`에서 이어서). 끝나면 `t1b_results.zip`(원문 없음).


## 1. 환경 준비

In [ ]:
SRC = {'store': '"""컬렉션 저장소 (SPEC §2.5, §2.6).\n\n관리자·개발자가 등록한 문서를 조각으로 나눠 임베딩과 함께 저장하고, 질문으로 검색한다.\n게이트웨이는 찾은 조각을 LLM 프롬프트에 넣고, **같은 조각을 판정 근거로** 쓴다(LLM이 본 근거 = 판정 근거).\n\n  Store 인터페이스\n    add(collection, text, title=\'\', doc_id=None) -> dict(doc_id, title, n_chunks, n_chars)\n    list(collection) -> [dict(doc_id, title, n_chunks, n_chars, created)]\n    delete(collection, doc_id) -> bool\n    collections() -> [dict(collection, n_docs)]\n    search(collection, query, k) -> [Evidence]       # 점수 내림차순\n\n처음 구현은 SQLite 파일 하나 + 임베딩 배열(코사인 유사도 전수 검색)이다. 규모가 커지면 같은 인터페이스로 벡터 DB로 바꾼다.\n임베더는 encode(texts) -> 정규화된 벡터 배열을 돌려주는 객체(sentence-transformers 호환)면 된다.\n"""\nimport sqlite3\nimport threading\nimport time\nimport uuid\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nSCHEMA = \'\'\'\nCREATE TABLE IF NOT EXISTS documents (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, title TEXT, text TEXT NOT NULL, created REAL NOT NULL,\n    PRIMARY KEY (collection, doc_id));\nCREATE TABLE IF NOT EXISTS chunks (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, idx INTEGER NOT NULL, text TEXT NOT NULL, emb BLOB NOT NULL,\n    PRIMARY KEY (collection, doc_id, idx));\n\'\'\'\n\n\n@dataclass\nclass Evidence:\n    doc_id: str\n    title: str\n    chunk_idx: int\n    text: str\n    score: float\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef split_words(text, n, overlap=0):\n    """n단어 조각(겹침 overlap단어). 빈 문서는 조각 없음."""\n    w = text.split()\n    if not w:\n        return []\n    step = max(1, n - overlap)\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w) - overlap, 1), step)]\n\n\nclass SQLiteStore:\n    def __init__(self, path, embedder, chunk_words=120, overlap=20):\n        self.path, self.embedder = path, embedder\n        self.chunk_words, self.overlap = chunk_words, overlap\n        self._db = sqlite3.connect(path, check_same_thread=False)\n        self._db.executescript(SCHEMA)\n        self._lock = threading.Lock()\n        self._cache = {}          # collection -> (meta 리스트, 임베딩 행렬)\n\n    def _embed(self, texts):\n        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)\n        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)\n\n    def add(self, collection, text, title=\'\', doc_id=None):\n        if not text or not text.strip():\n            raise ValueError(\'document text is empty\')\n        doc_id = doc_id or uuid.uuid4().hex[:16]\n        chunks = split_words(text, self.chunk_words, self.overlap)\n        emb = self._embed(chunks)\n        with self._lock, self._db:\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))   # 같은 id면 교체\n            self._db.execute(\'INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?)\',\n                             (collection, doc_id, title or \'\', text, time.time()))\n            self._db.executemany(\'INSERT INTO chunks VALUES (?,?,?,?,?)\',\n                                 [(collection, doc_id, i, c, e.tobytes()) for i, (c, e) in enumerate(zip(chunks, emb))])\n            self._cache.pop(collection, None)\n        return {\'doc_id\': doc_id, \'title\': title or \'\', \'n_chunks\': len(chunks), \'n_chars\': len(text)}\n\n    def list(self, collection):\n        with self._lock:\n            rows = self._db.execute(\n                \'SELECT d.doc_id, d.title, length(d.text), d.created, \'\n                \'(SELECT count(*) FROM chunks c WHERE c.collection=d.collection AND c.doc_id=d.doc_id) \'\n                \'FROM documents d WHERE d.collection=? ORDER BY d.created\', (collection,)).fetchall()\n        return [{\'doc_id\': r[0], \'title\': r[1], \'n_chars\': r[2], \'created\': r[3], \'n_chunks\': r[4]} for r in rows]\n\n    def delete(self, collection, doc_id):\n        with self._lock, self._db:\n            n = self._db.execute(\'DELETE FROM documents WHERE collection=? AND doc_id=?\', (collection, doc_id)).rowcount\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))\n            self._cache.pop(collection, None)\n        return n > 0\n\n    def collections(self):\n        with self._lock:\n            rows = self._db.execute(\'SELECT collection, count(*) FROM documents GROUP BY collection ORDER BY collection\').fetchall()\n        return [{\'collection\': c, \'n_docs\': n} for c, n in rows]\n\n    def exists(self, collection):\n        with self._lock:\n            return self._db.execute(\'SELECT 1 FROM documents WHERE collection=? LIMIT 1\', (collection,)).fetchone() is not None\n\n    def _matrix(self, collection):\n        with self._lock:\n            if collection not in self._cache:\n                rows = self._db.execute(\n                    \'SELECT c.doc_id, d.title, c.idx, c.text, c.emb FROM chunks c JOIN documents d \'\n                    \'ON c.collection=d.collection AND c.doc_id=d.doc_id WHERE c.collection=? ORDER BY d.created, c.idx\',\n                    (collection,)).fetchall()\n                meta = [r[:4] for r in rows]\n                mat = np.stack([np.frombuffer(r[4], dtype=np.float32) for r in rows]) if rows else np.zeros((0, 1), np.float32)\n                self._cache[collection] = (meta, mat)\n            return self._cache[collection]\n\n    def search(self, collection, query, k):\n        meta, mat = self._matrix(collection)\n        if not meta or not query.strip():\n            return []\n        sims = mat @ self._embed([query])[0]\n        top = np.argsort(-sims)[:k]\n        return [Evidence(meta[i][0], meta[i][1], meta[i][2], meta[i][3], float(sims[i])) for i in top]\n\n    def count_docs(self, prefix):\n        """이름이 prefix로 시작하는 컬렉션들의 문서 수 (운영 모드 사용자별 한도, SPEC §10.2)."""\n        with self._lock:\n            return self._db.execute("SELECT count(*) FROM documents WHERE substr(collection, 1, ?)=?",\n                                    (len(prefix), prefix)).fetchone()[0]\n\n    def doc_exists(self, collection, doc_id):\n        with self._lock:\n            return self._db.execute(\'SELECT 1 FROM documents WHERE collection=? AND doc_id=?\',\n                                    (collection, doc_id)).fetchone() is not None\n\n    def purge(self, prefix):\n        """이름이 prefix로 시작하는 컬렉션을 모두 지운다 (키 폐기 --purge). 지운 문서 수."""\n        with self._lock, self._db:\n            n = self._db.execute(\'DELETE FROM documents WHERE substr(collection, 1, ?)=?\', (len(prefix), prefix)).rowcount\n            self._db.execute(\'DELETE FROM chunks WHERE substr(collection, 1, ?)=?\', (len(prefix), prefix))\n            self._cache.clear()\n        return n\n\n    def close(self):\n        self._db.close()\n', 'context': '"""요청에서 근거 문서(RAG 컨텍스트)와 검증 옵션을 꺼낸다.\n\n우선순위\n1. 최상위 필드 `verification.context` (문자열 또는 문자열 리스트)\n   - OpenAI SDK에서는 `extra_body={"verification": {...}}` 로 보낸다.\n2. 메시지 본문 안의 `<context>...</context>` 태그 (RAG 서비스가 흔히 쓰는 형식)\n3. `verification.collection` — 게이트웨이에 등록된 컬렉션에서 검색 (app이 처리, SPEC §2.6)\n4. 모두 없으면 검증하지 않는다 (mode=off).\n\n`verification` 필드는 실제 LLM API가 모르는 필드이므로 전달 전에 제거한다.\n"""\nimport copy\nimport re\n\nCTX_TAG = re.compile(r\'<context>(.*?)</context>\', re.S | re.I)\nMODES = {\'pass\', \'hold\', \'off\'}\nACTIONS = {\'flag\', \'cut\'}\n\n\ndef _content_text(content):\n    if isinstance(content, str):\n        return content\n    if isinstance(content, list):          # [{"type":"text","text":...}, ...]\n        return \'\\n\'.join(p.get(\'text\', \'\') for p in content if isinstance(p, dict))\n    return \'\'\n\n\nRAG_INSTRUCTION = (\'Answer using only the reference documents below. \'\n                   \'If they do not contain the answer, say that you do not know.\')\n\n\ndef last_user_text(body):\n    users = [m for m in body.get(\'messages\', []) if m.get(\'role\') == \'user\']\n    return _content_text(users[-1].get(\'content\')) if users else \'\'\n\n\ndef rag_messages(messages, evidence):\n    """근거 조각을 담은 시스템 메시지를 맨 앞에 더한다. 서비스가 보낸 메시지는 바꾸지 않는다 (SPEC §2.6)."""\n    block = \'\\n\\n\'.join(f\'[{i + 1}] {e.text}\' for i, e in enumerate(evidence))\n    return [{\'role\': \'system\', \'content\': f\'{RAG_INSTRUCTION}\\n\\n<documents>\\n{block}\\n</documents>\'}] + list(messages)\n\n\ndef strip_context_tags(text):\n    """질문 텍스트: <context> 태그 내용을 뺀 나머지 (근거 다이어트 검색어, SPEC §2.9)."""\n    return CTX_TAG.sub(\' \', text or \'\').strip()\n\n\ndef replace_context_tags(messages, new_ref):\n    """첫 <context> 태그 내용을 new_ref로 바꾸고 나머지 태그는 지운다. 다른 내용은 그대로 (SPEC §2.9)."""\n    out, done = [], False\n    for m in messages:\n        c = m.get(\'content\')\n        if isinstance(c, str) and CTX_TAG.search(c):\n            def sub(mt):\n                nonlocal done\n                if done:\n                    return \'\'\n                done = True\n                return f\'<context>{new_ref}</context>\'\n            m = {**m, \'content\': CTX_TAG.sub(sub, c)}\n        out.append(m)\n    return out\n\n\ndef rag_reference(evidence):\n    """판정 근거 = LLM에 넣은 조각과 같은 텍스트."""\n    return \'\\n\\n\'.join(e.text for e in evidence)\n\n\ndef extract(body, default_mode=\'pass\', default_action=\'flag\', default_stride=0, with_stride=False, with_opts=False):\n    """반환: (upstream으로 보낼 body, reference 문자열 또는 None, mode, action[, stride][, opts])\n    with_stride=True면 앞부분 재판정 간격(stride)도 함께 돌려준다.\n    with_opts=True면 원래 verification 옵션도 돌려준다(collection 등). collection이 있고 근거가 없으면 mode를 off로 바꾸지 않는다."""\n    body = copy.deepcopy(body)\n    opts = body.pop(\'verification\', None) or {}\n\n    ref = opts.get(\'context\')\n    if isinstance(ref, list):\n        ref = \'\\n\\n\'.join(str(x) for x in ref)\n    if not ref:\n        found = []\n        for m in body.get(\'messages\', []):\n            found += CTX_TAG.findall(_content_text(m.get(\'content\')))\n        ref = \'\\n\\n\'.join(s.strip() for s in found) or None\n\n    mode = opts.get(\'mode\', default_mode)\n    action = opts.get(\'action\', default_action)\n    if mode not in MODES:\n        raise ValueError(f\'verification.mode must be one of {sorted(MODES)}\')\n    if action not in ACTIONS:\n        raise ValueError(f\'verification.action must be one of {sorted(ACTIONS)}\')\n    stride = opts.get(\'stride\', default_stride)\n    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 0:\n        raise ValueError(\'verification.stride must be a non-negative integer\')\n    coll = opts.get(\'collection\')\n    if coll is not None and (not isinstance(coll, str) or not coll):\n        raise ValueError(\'verification.collection must be a non-empty string\')\n    tr = opts.get(\'trim\')\n    if tr is not None and not (isinstance(tr, bool) or (isinstance(tr, dict) and set(tr) <= {\'top_k\'}\n                                                         and isinstance(tr.get(\'top_k\'), int) and not isinstance(tr.get(\'top_k\'), bool)\n                                                         and 1 <= tr[\'top_k\'] <= 32)):\n        raise ValueError(\'verification.trim must be true/false or {"top_k": 1..32}\')\n    pc = opts.get(\'post_check\')\n    if pc is not None and not isinstance(pc, bool):\n        raise ValueError(\'verification.post_check must be a boolean\')\n    if not ref and not coll:\n        mode = \'off\'\n    out = (body, ref, mode, action) + ((stride,) if with_stride else ()) + ((opts,) if with_opts else ())\n    return out\n', 'trim': '"""근거 다이어트 (SPEC §2.9): 긴 근거에서 질문과 관련된 조각만 골라 LLM에 보내고 같은 조각으로 판정한다.\n\n조각 나누기는 컬렉션(store.split_words)과 같고, 임베더도 컬렉션 검색과 같은 것을 쓴다.\n고른 조각은 원문 순서대로 돌려준다(문서 흐름 유지).\n"""\nimport numpy as np\n\nfrom .store import split_words\n\n\nclass Trimmer:\n    def __init__(self, embedder, chunk_words=120, overlap=20):\n        self.embedder, self.chunk_words, self.overlap = embedder, chunk_words, overlap\n\n    def _embed(self, texts):\n        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)\n        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)\n\n    def select(self, reference, query, k):\n        """(고른 조각 리스트(원문 순서), 전체 조각 수). 조각이 k개 이하이거나 질문이 비면 전체를 그대로."""\n        chunks = split_words(reference, self.chunk_words, self.overlap) or [reference]\n        if len(chunks) <= k or not (query or \'\').strip():\n            return chunks if len(chunks) > 1 else [reference], len(chunks)\n        sims = self._embed(chunks) @ self._embed([query])[0]\n        top = sorted(np.argsort(-sims)[:k].tolist())\n        return [chunks[i] for i in top], len(chunks)\n'}
import os, sys, json, time, random, subprocess, shutil
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'anthropic', 'sentence-transformers', 'tiktoken', 'pysbd'], check=True)
subprocess.run('git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || true', shell=True)
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/hg/hallugate', exist_ok=True)
open('/content/hg/hallugate/__init__.py', 'w').write('')
for n, s in SRC.items(): open(f'/content/hg/hallugate/{n}.py', 'w').write(s)
sys.path.insert(0, '/content/hg')
OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
T1B_DIR = os.path.join(OUT, 't1b'); os.makedirs(T1B_DIR, exist_ok=True)
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None

## 2. 질문 200개 (T1 질문 제외) — 긴 근거

In [ ]:
import pandas as pd
N_ITEMS, N_DISTRACT, SEED = 200, 7, 2037          # 사전 고정 (SSOT §6 T1)

def train_qa_sources(responses, sources):
    # RAGTruth train 분할 응답만 있는 QA 근거 (test 분할 근거는 쓰지 않음)
    split_of = {}
    for r in responses:
        split_of.setdefault(r['source_id'], set()).add(r['split'])
    out = [s for s in sources if s['task_type'] == 'QA' and split_of.get(s['source_id']) == {'train'}]
    assert all(split_of[s['source_id']] == {'train'} for s in out)
    return sorted(out, key=lambda s: str(s['source_id']))

def make_items(pool):
    rng = random.Random(SEED)
    picked = rng.sample(pool, min(N_ITEMS, len(pool)))
    items = []
    for i, s in enumerate(picked):
        others = rng.sample([o for o in pool if o['source_id'] != s['source_id']], N_DISTRACT)
        blocks = [s['source_info']['passages']] + [o['source_info']['passages'] for o in others]
        order = list(range(len(blocks))); rng.shuffle(order)
        items.append({'i': i, 'sid': s['source_id'], 'q': s['source_info']['question'],
                      'own': s['source_info']['passages'],
                      'ref_L': '\n\n'.join(blocks[k] for k in order), 'ref_S': s['source_info']['passages']})
    return items

N_ITEMS_B, SEED_B = 200, 2038                        # 사전 고정 (SSOT §6 T1b)

def make_items_b(pool, exclude):
    # T1에서 결과를 본 질문(exclude)은 빼고 새로 뽑는다. 근거는 T1의 L과 같은 방식
    rng = random.Random(SEED_B)
    cand = [s for s in pool if s['source_id'] not in exclude]
    picked = rng.sample(cand, min(N_ITEMS_B, len(cand)))
    items = []
    for i, s in enumerate(picked):
        others = rng.sample([o for o in pool if o['source_id'] != s['source_id']], N_DISTRACT)
        blocks = [s['source_info']['passages']] + [o['source_info']['passages'] for o in others]
        order = list(range(len(blocks))); rng.shuffle(order)
        items.append({'i': i, 'sid': s['source_id'], 'q': s['source_info']['question'], 'own': s['source_info']['passages'],
                      'ref_L': '\n\n'.join(blocks[k] for k in order), 'ref_S': s['source_info']['passages']})
    assert not {x['sid'] for x in items} & set(exclude), 'T1 질문이 섞임'
    return items

BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources = [json.loads(l) for l in open(f'{BASE}/source_info.jsonl')]
POOL = train_qa_sources(responses, sources)
T1_SIDS = {x['sid'] for x in make_items(POOL)}             # T1에서 결과를 본 질문 (재현)
ITEMS = make_items_b(POOL, T1_SIDS)
print(f'train QA {len(POOL)}개, T1 제외 후 {len(ITEMS)}개, 긴 근거 평균 {np.mean([len(x["ref_L"]) for x in ITEMS]):.0f}자')

## 3. 다이어트 (게이트웨이 Trimmer)

In [ ]:
from hallugate.trim import Trimmer
from hallugate.context import rag_messages
from types import SimpleNamespace
TOP_K, TOP_KS_REPORT = 4, [2, 4, 8]
GEN_MODEL, LABELER = 'claude-haiku-5-5', 'claude-opus-5-5'

def ngrams(t, n=8):
    w = t.lower().split()
    return {tuple(w[k:k + n]) for k in range(max(len(w) - n + 1, 0))}

def trim_all(items, trimmer, ntok):
    rows = []
    for it in items:
        own = ngrams(it['own'])
        for cond in ['L', 'S']:
            ref = it[f'ref_{cond}']
            row = {'i': it['i'], 'cond': cond, 'ref_tok_full': ntok(ref)}
            for k in TOP_KS_REPORT:
                kept, total = trimmer.select(ref, it['q'], k)
                new = '\n\n'.join(kept) if total > k else ref
                row[f'ref_tok_k{k}'] = ntok(new)
                row[f'own_recall_k{k}'] = len(ngrams(new) & own) / max(len(own), 1)   # 원래 지문이 얼마나 남았나 (보고)
                if k == TOP_K:
                    it[f'trim_{cond}'] = [new] if total <= k else kept
                    row['chunks_total'] = total
            rows.append(row)
    return pd.DataFrame(rows)

def gen_params(q, chunks):
    msgs = rag_messages([{'role': 'user', 'content': q}], [SimpleNamespace(text=c) for c in chunks])   # 게이트웨이와 같은 RAG 지시문
    return {'model': GEN_MODEL, 'max_tokens': 512, 'system': msgs[0]['content'], 'messages': msgs[1:]}

def gen_requests(items):
    R = {}
    for it in items:
        for cond in ['L', 'S']:
            R[f'g-{it["i"]}-{cond}-full'] = gen_params(it['q'], [it[f'ref_{cond}']])
            R[f'g-{it["i"]}-{cond}-trim'] = gen_params(it['q'], it[f'trim_{cond}'])
    return R

PAIR_SYSTEM = '''You compare two answers to the same question. The PASSAGES are the only source of truth.
Judge which answer better answers the QUESTION using correct information supported by the PASSAGES.
Prefer the answer that answers more completely and correctly. An answer that says it does not know when the PASSAGES do contain the answer is worse.
If both are equally good (or equally bad), say "same".
Return ONLY a JSON object: {"better": "1" | "2" | "same", "reason": "at most 20 words"}'''

def pair_params(it, cond, a1, a2):
    u = f"QUESTION: {it['q']}\n\nPASSAGES:\n<<<\n{it['own']}\n>>>\n\nANSWER 1:\n<<<\n{a1}\n>>>\n\nANSWER 2:\n<<<\n{a2}\n>>>"
    return {'model': LABELER, 'max_tokens': 300, 'system': PAIR_SYSTEM, 'messages': [{'role': 'user', 'content': u}]}

def trim_first(i, cond):
    # 다이어트 답을 1번에 둘지 (쌍마다 고정된 무작위, 판정자에게 가림)
    return random.Random(f'{SEED}-{i}-{cond}').random() < 0.5

def parse_pair(text):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        b = str(d.get('better', '')).strip().lower()
        return b if b in ('1', '2', 'same') else None
    except Exception:
        return None

def judge_requests(items, G):
    P, F = {}, {}
    for it in items:
        for cond in ['L', 'S']:
            gf, gt = G.get(f'g-{it["i"]}-{cond}-full'), G.get(f'g-{it["i"]}-{cond}-trim')
            if not (gf and gt and gen_status(gf) == 'valid' and gen_status(gt) == 'valid'):
                continue
            a_full, a_trim = gf['text'], gt['text']
            first_trim = trim_first(it['i'], cond)
            P[f'p-{it["i"]}-{cond}'] = pair_params(it, cond, *((a_trim, a_full) if first_trim else (a_full, a_trim)))
            case = {'task': 'QA', 'ref_label': f"Question: {it['q']}\n{it['own']}"}
            for v, a in [('full', a_full), ('trim', a_trim)]:
                p = label_params(case, a); p['model'] = LABELER
                F[f'f-{it["i"]}-{cond}-{v}'] = p
    return P, F

import tiktoken
from sentence_transformers import SentenceTransformer
ENC = tiktoken.get_encoding('cl100k_base')
ntok = lambda t: len(ENC.encode(t, disallowed_special=()))
T = trim_all(ITEMS, Trimmer(SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu'), 120, 20), ntok)
d = T[T.cond == 'L']
print('근거 토큰 감소 중앙값(k4)', round(float((1 - d.ref_tok_k4 / d.ref_tok_full).median()), 3), '| 원래 지문 재현(k4)', round(float(d.own_recall_k4.mean()), 3))

## 4. 생성 (Haiku, temperature 0) — 형식 확인 → 비용 추정 → 배치

In [ ]:
import pysbd
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)
SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

def sentences(text):
    # (시작, 끝, 문장) — 판정 지점(points_of)과 같은 pysbd 분할
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        out.append((a, sp.start + len(sp.sent.rstrip()), sp.sent.strip()))
    return out

def label_msg(c, text):
    ss = sentences(text)
    lines = '\n'.join(f'[{i + 1}] {s}' for i, (_, _, s) in enumerate(ss))
    return f"TASK: {TASK_DESC[c['task']]}\n\nREFERENCE:\n<<<\n{c['ref_label']}\n>>>\n\nRESPONSE ({len(ss)} numbered sentences):\n{lines}"

def label_params(c, text):
    # R4와 같은 요청 (모델·max_tokens·지시문)
    return {'model': LABELER, 'max_tokens': 4096, 'system': SYSTEM, 'messages': [{'role': 'user', 'content': label_msg(c, text)}]}

def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

def score_response(jF, jI, refs, text, fn):
    # 판정 지점(문장마다 8토큰째 + 문장 끝)마다 (문장 시작, 지점 끝, F1 base, I2, 조합)
    out = []
    for a, cut in points_of(text):
        t = text[a:cut].strip()
        if not t: continue
        sf, si = fn(jF, jI, refs[0], refs[1], {'text': t, 'prior': text[:a]})
        out.append((a, cut, sf, si, combine(sf, si)))
    return out

def gen_status(g):
    if g is None: return 'nogen'
    if g['stop'] == 'refusal': return 'refusal'
    if not g['text'].strip() or not sentences(g['text']): return 'empty'
    return 'valid'

import pandas as pd

def wilson(k, n, z=1.96):
    if n == 0: return float('nan'), float('nan'), float('nan')
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def conformal(maxes, alpha=0.10):
    # cal-L 노트북(conformal_flag)과 같은 등각 보정: 멀쩡한 응답 최대 점수의 ceil((n+1)(1-α))번째 + 1e-6
    mx = np.sort(np.asarray(maxes, float)); n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def response_rows(cases, gens, labels, scores, thr_c, thr_f):
    rows = []
    for cid, c in cases.items():
        g = gens.get(cid); st = gen_status(g)
        r = {'cid': cid, 'sid': c['sid'], 'task': c['task'], 'gen': c['gen'], 'flow': c['flow'], 'gold_in': c['gold_in'],
             'stop': g['stop'] if g else None, 'status': st}
        if st == 'valid':
            ss = sentences(g['text']); lab = labels.get(cid)
            r['status'] = 'ok' if lab is not None else 'nolabel'
            pts = scores.get(cid, [])
            r['n_sent'], r['n_pts'] = len(ss), len(pts)
            r['max_c'] = max([p[4] for p in pts], default=-np.inf)
            r['max_f'] = max([p[2] for p in pts], default=-np.inf)
            r['flag_c'], r['flag_f'] = bool(r['max_c'] >= thr_c), bool(r['max_f'] >= thr_f)
            first = next((p[0] for p in pts if p[4] >= thr_c), None)
            r['first_alarm_a'] = first
            if lab is not None:
                ls = [lab[i + 1]['label'] for i in range(len(ss))]
                r['n_H'], r['n_A'] = ls.count('H'), ls.count('A')
                r['clean'], r['strict_clean'] = r['n_H'] == 0, all(l == 'S' for l in ls)
                hs = [ss[i][0] for i, l in enumerate(ls) if l == 'H']
                r['first_H_a'] = hs[0] if hs else None
                r['strict_det'] = bool(hs and first is not None and first >= hs[0])
        rows.append(r)
    return pd.DataFrame(rows)

def rate_line(df, col='flag_c'):
    k, n = int(df[col].sum()), len(df)
    p, lo, hi = wilson(k, n)
    return {'k': k, 'n': n, 'rate': p, 'lo': lo, 'hi': hi}

def fmt(d):
    return f"{d['rate']:.3f} [{d['lo']:.3f}, {d['hi']:.3f}] ({d['k']}/{d['n']})" if d['n'] else '—'

def report(df, thr_name=''):
    ok = df[df.status == 'ok'].copy()
    for col in ['clean', 'strict_clean', 'flag_c', 'flag_f', 'strict_det']:
        ok[col] = ok[col].astype(bool)
    clean, hal = ok[ok.clean], ok[~ok.clean]
    out = {'status': df.status.value_counts().to_dict(), 'max_tokens_stops': int((df.stop == 'max_tokens').sum()),
           'clean_combo': rate_line(clean), 'clean_f1': rate_line(clean, 'flag_f'),
           'strict_clean_combo': rate_line(ok[ok.strict_clean]),
           'hal_any': rate_line(hal), 'hal_strict': rate_line(hal, 'strict_det'),
           'flag_precision_H': rate_line(ok[ok.flag_c], 'clean').copy(),
           'by_gen': {g: rate_line(d) for g, d in clean.groupby('gen')},
           'by_task': {t: rate_line(d) for t, d in clean.groupby('task')},
           'by_gen_task': {f'{g}|{t}': rate_line(d) for (g, t), d in clean.groupby(['gen', 'task'])},
           'hal_rate_by_gen': {g: float((~d.clean).mean()) for g, d in ok.groupby('gen')}}
    fp = out['flag_precision_H']                     # 표시된 응답 중 멀쩡한 비율 → H 비율로 바꿈
    if fp['n']: fp.update(k=fp['n'] - fp['k'], rate=1 - fp['rate'], lo=1 - fp['hi'], hi=1 - fp['lo'])
    qa = df[(df.flow == 'collection') & (df.gen == 'haiku')]
    out['qa_gold_in'] = float(qa.gold_in.astype(float).mean()) if len(qa) else None
    return out

import anthropic
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=3)
PRICE = {'claude-haiku-5-5': (0.05, 0.25), 'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}   # 배치 $/백만 토큰 (2026-11)

def text_of(m): return ''.join(b.text for b in m.content if getattr(b, 'type', '') == 'text')
def rec(m): return {'text': text_of(m), 'stop': m.stop_reason, 'model': m.model, 'in': m.usage.input_tokens, 'out': m.usage.output_tokens}
def cost(res): return sum(r['in'] / 1e6 * PRICE.get(r['model'], (2, 10))[0] + r['out'] / 1e6 * PRICE.get(r['model'], (2, 10))[1] for r in res.values())

def run_batch(reqs, path, chunk=400):
    # reqs: cid -> 요청. 결과: cid -> rec. 다시 실행하면 제출한 묶음·받은 결과는 건너뛴다
    res = load_json(path) or {}
    bpath = path.replace('.json', '_batches.json')
    B = load_json(bpath) or {'chunks': []}
    sent = {c for ch in B['chunks'] for c in ch['cids']}
    todo = [c for c in reqs if c not in sent and c not in res]
    for k in range(0, len(todo), chunk):
        part = todo[k:k + chunk]
        b = A.messages.batches.create(requests=[{'custom_id': c, 'params': reqs[c]} for c in part])
        B['chunks'].append({'id': b.id, 'cids': part, 'done': False}); save_json(B, bpath)
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    for ch in B['chunks']:
        if ch.get('done'): continue
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded': res[r.custom_id] = rec(r.result.message)
        ch['done'] = True; save_json(B, bpath); save_json(res, path)
    miss = [c for c in reqs if c not in res]
    for c in miss[:100]:                                   # 배치 실패분은 바로 요청 1회
        try: res[c] = rec(A.messages.create(**reqs[c]))
        except Exception as e: print(f'  {c}: {type(e).__name__} {str(e)[:150]}'); break
    save_json(res, path)
    print(f'  결과 {len(res)}/{len(reqs)} | 사용량 약 ${cost({c: res[c] for c in reqs if c in res}):.2f}')
    return res


ARMS = ['full', 'full2', 'trim']                     # full2 = 같은 입력 재생성 대조군

def gen_params_b(q, chunks, use_temp):
    p = gen_params(q, chunks); p['max_tokens'] = 1024
    if use_temp: p['temperature'] = 0.0
    return p

def gen_requests_b(items, use_temp=True):
    R = {}
    for it in items:
        R[f'g-{it["i"]}-full'] = gen_params_b(it['q'], [it['ref_L']], use_temp)
        R[f'g-{it["i"]}-full2'] = gen_params_b(it['q'], [it['ref_L']], use_temp)
        R[f'g-{it["i"]}-trim'] = gen_params_b(it['q'], it['trim_L'], use_temp)
    return R

GRADE_SYSTEM = '''You grade ONE answer to a question. The PASSAGES are the only source of truth.
"yes": the answer correctly answers the QUESTION with information supported by the PASSAGES (minor omissions are fine).
"partial": the answer is on the right track but misses an important part of what the PASSAGES say, or mixes correct and incorrect points.
"no": the answer does not answer the question, is wrong, or says the information is unavailable while the PASSAGES do contain it.
If the PASSAGES truly do not contain the answer, an answer that says so is "yes".
Return ONLY a JSON object: {"grade": "yes" | "partial" | "no", "reason": "at most 20 words"}'''

def grade_params(it, ans):
    u = f"QUESTION: {it['q']}\n\nPASSAGES:\n<<<\n{it['own']}\n>>>\n\nANSWER:\n<<<\n{ans}\n>>>"
    return {'model': LABELER, 'max_tokens': 300, 'system': GRADE_SYSTEM, 'messages': [{'role': 'user', 'content': u}]}

def parse_grade(text):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        g = str(d.get('grade', '')).strip().lower()
        return g if g in ('yes', 'partial', 'no') else None
    except Exception:
        return None

def judge_requests_b(items, G):
    # 세 팔이 모두 정상 생성된 질문만. 평가 요청에는 어느 팔인지 드러나지 않는다(답 하나씩)
    Q, F = {}, {}
    for it in items:
        gs = {a: G.get(f'g-{it["i"]}-{a}') for a in ARMS}
        if not all(g and gen_status(g) == 'valid' for g in gs.values()):
            continue
        case = {'task': 'QA', 'ref_label': f"Question: {it['q']}\n{it['own']}"}
        for a, g in gs.items():
            Q[f'q-{it["i"]}-{a}'] = grade_params(it, g['text'])
            p = label_params(case, g['text']); p['model'] = LABELER
            F[f'f-{it["i"]}-{a}'] = p
    return Q, F

USE_TEMP = True
_it = ITEMS[0]
try:                                                   # temperature 0 형식 확인 (거부되면 빼고 진행, 사전 고정)
    _r = rec(A.messages.create(**gen_params_b(_it['q'], _it['trim_L'], True)))
    print('형식 확인: temperature 0 사용 | 입력', _r['in'], '출력', _r['out'])
except Exception as e:
    if 'temperature' in str(e).lower():
        USE_TEMP = False; print('temperature 거부 → 기본값으로 진행:', str(e)[:200])
    else:
        raise
BUDGET_USD = 12.0
REQ_G = gen_requests_b(ITEMS, USE_TEMP)
def est(reqs, out_tok):
    tot = 0.0
    for p in reqs.values():
        n_in = (len(p.get('system', '')) + sum(len(m['content']) for m in p['messages'])) / 3.5
        pin, pout = PRICE[p['model']]; tot += n_in / 1e6 * pin + out_tok / 1e6 * pout
    return tot
EST_G = est(REQ_G, 400)
EST_J = len(ITEMS) * 3 * 2 * (2200 * PRICE[LABELER][0] + 500 * PRICE[LABELER][1]) / 1e6
print(f'생성 {len(REQ_G)}건 예상 ${EST_G:.2f} + 평가 약 ${EST_J:.2f} (상한 ${BUDGET_USD})')
assert EST_G + EST_J <= BUDGET_USD, '예상 비용이 상한을 넘음 — 멈춤'
G = run_batch(REQ_G, os.path.join(T1B_DIR, 'gen.json'))
print('생성 상태', pd.Series([gen_status(G.get(k)) for k in REQ_G]).value_counts().to_dict(),
      '| 잘림', sum(1 for k in REQ_G if G.get(k, {}).get('stop') == 'max_tokens'))

## 5. 평가 (Opus, 답마다 따로): 정답성 + 충실성

In [ ]:
REQ_Q, REQ_F = judge_requests_b(ITEMS, G)
QR = run_batch(REQ_Q, os.path.join(T1B_DIR, 'grade.json'))
FR = run_batch(REQ_F, os.path.join(T1B_DIR, 'faith.json'))
print(f'정답성 {len(QR)}/{len(REQ_Q)}, 충실성 {len(FR)}/{len(REQ_F)}')

## 6. 기준 판정

In [ ]:
def outcome(i, cond, better):
    if better is None: return None
    if better == 'same': return 'SAME'
    trim_is_1 = trim_first(i, cond)
    return 'BETTER' if (better == '1') == trim_is_1 else 'WORSE'

def hal(res, key, text):
    r = res.get(key)
    if r is None: return None
    labs, ok = parse(r['text'], len(sentences(text)))
    return ('H' in [v['label'] for v in labs.values()]) if ok else None

def build_rows(items, G, PR, FR, T):
    rows = []
    for it in items:
        for cond in ['L', 'S']:
            kp = f'p-{it["i"]}-{cond}'
            if kp not in PR: continue
            gf, gt = G[f'g-{it["i"]}-{cond}-full'], G[f'g-{it["i"]}-{cond}-trim']
            rows.append({'i': it['i'], 'cond': cond, 'in_full': gf['in'], 'in_trim': gt['in'],
                         'out_full': gf['out'], 'out_trim': gt['out'],
                         'outcome': outcome(it['i'], cond, parse_pair(PR[kp]['text'])), 'trim_first': trim_first(it['i'], cond),
                         'hal_full': hal(FR, f'f-{it["i"]}-{cond}-full', gf['text']),
                         'hal_trim': hal(FR, f'f-{it["i"]}-{cond}-trim', gt['text'])})
    D = pd.DataFrame(rows).merge(T, on=['i', 'cond'], how='left')
    D['valid'] = D.outcome.notna() & D.hal_full.notna() & D.hal_trim.notna()
    return D

def paired_boot(d, n=2000, seed=SEED):
    rng = np.random.default_rng(seed); x = (d.hal_trim.astype(float) - d.hal_full.astype(float)).values; vals = []
    for _ in range(n):
        vals.append(x[rng.integers(0, len(x), len(x))].mean())
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def judge_t1(D):
    out = {}
    for cond in ['L', 'S']:
        d = D[(D.cond == cond) & D.valid]
        red = 1 - d.in_trim / d.in_full
        w = int((d.outcome == 'WORSE').sum()); n = len(d)
        p, lo, hi = wilson(w, n)
        dh = float(d.hal_trim.mean() - d.hal_full.mean()) if n else float('nan')
        out[cond] = {'n': n, 'input_reduction_median': float(red.median()) if n else None,
                     'ref_token_reduction_median': float((1 - d.ref_tok_k4 / d.ref_tok_full).median()) if n else None,
                     'worse': p, 'worse_ci': [lo, hi], 'same': float((d.outcome == 'SAME').mean()) if n else None,
                     'better': float((d.outcome == 'BETTER').mean()) if n else None,
                     'hal_full': float(d.hal_full.mean()) if n else None, 'hal_trim': float(d.hal_trim.mean()) if n else None,
                     'hal_diff': dh, 'hal_diff_ci': paired_boot(d) if n else None,
                     'own_recall_k': {k: float(d[f'own_recall_k{k}'].mean()) for k in TOP_KS_REPORT} if n else None,
                     'ref_tok_k': {k: float((1 - d[f'ref_tok_k{k}'] / d.ref_tok_full).median()) for k in TOP_KS_REPORT} if n else None,
                     'worse_by_order': {str(t): float((g.outcome == 'WORSE').mean()) for t, g in d.groupby('trim_first')} if n else None}
    L = out['L']
    out['T1_0'] = bool(out['L']['n'] >= 180 and out['S']['n'] >= 180)
    out['T1_1'] = bool(L['n'] and L['input_reduction_median'] >= 0.40)
    out['T1_2'] = bool(L['n'] and L['worse'] <= 0.10 and L['worse_ci'][1] <= 0.15)
    out['T1_3'] = bool(L['n'] and L['hal_diff'] <= 0.03 and L['hal_diff_ci'][1] < 0.08)
    out['T1'] = out['T1_0'] and out['T1_1'] and out['T1_2'] and out['T1_3']
    return out

def build_rows_b(items, G, QR, FR, T):
    rows = []
    for it in items:
        row = {'i': it['i']}
        ok = True
        for a in ARMS:
            g = G.get(f'g-{it["i"]}-{a}')
            if not g or gen_status(g) != 'valid':
                ok = False; break
            q = QR.get(f'q-{it["i"]}-{a}')
            row[f'in_{a}'], row[f'out_{a}'] = g['in'], g['out']
            row[f'trunc_{a}'] = g.get('stop') == 'max_tokens'
            row[f'grade_{a}'] = parse_grade(q['text']) if q else None
            row[f'hal_{a}'] = hal(FR, f'f-{it["i"]}-{a}', g['text'])
        if ok:
            rows.append(row)
    D = pd.DataFrame(rows).merge(T[T.cond == 'L'].drop(columns=['cond']), on='i', how='left')
    D['valid'] = D[[f'grade_{a}' for a in ARMS] + [f'hal_{a}' for a in ARMS]].notna().all(axis=1)
    return D

def paired_boot_b(x, n=2000, seed=SEED_B):
    rng = np.random.default_rng(seed); x = np.asarray(x, float)
    vals = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(n)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def judge_t1b(D):
    d = D[D.valid]; n = len(d)
    yes = {a: (d[f'grade_{a}'] == 'yes').astype(float) for a in ARMS}
    hal_ = {a: d[f'hal_{a}'].astype(float) for a in ARMS}
    out = {'n': n,
           'yes': {a: float(yes[a].mean()) for a in ARMS},
           'partial': {a: float((d[f'grade_{a}'] == 'partial').mean()) for a in ARMS},
           'no': {a: float((d[f'grade_{a}'] == 'no').mean()) for a in ARMS},
           'hal': {a: float(hal_[a].mean()) for a in ARMS},
           'trunc': {a: float(d[f'trunc_{a}'].mean()) for a in ARMS},
           'input_reduction_median': float((1 - d.in_trim / d.in_full).median()),
           'own_recall_k4': float(d.own_recall_k4.mean())}
    out['control_yes_diff'] = float((yes['full2'] - yes['full']).mean())
    out['control_yes_ci'] = paired_boot_b(yes['full2'] - yes['full'])
    out['control_hal_diff'] = float((hal_['full2'] - hal_['full']).mean())
    out['yes_diff'] = float((yes['trim'] - yes['full']).mean())
    out['yes_diff_ci'] = paired_boot_b(yes['trim'] - yes['full'])
    out['hal_diff'] = float((hal_['trim'] - hal_['full']).mean())
    out['hal_diff_ci'] = paired_boot_b(hal_['trim'] - hal_['full'])
    out['T1b_0'] = bool(n >= 180 and abs(out['control_yes_diff']) <= 0.05)
    out['T1b_1'] = bool(out['input_reduction_median'] >= 0.40)
    out['T1b_2'] = bool(out['yes_diff'] >= -0.05 and out['yes_diff_ci'][0] >= -0.10)
    out['T1b_3'] = bool(out['hal_diff'] <= 0.03 and out['hal_diff_ci'][1] < 0.08)
    out['T1b'] = out['T1b_0'] and out['T1b_1'] and out['T1b_2'] and out['T1b_3']
    return out

D = build_rows_b(ITEMS, G, QR, FR, T)
R = judge_t1b(D)
print(f"유효 {R['n']} | yes: full {R['yes']['full']:.3f}, full2 {R['yes']['full2']:.3f}, trim {R['yes']['trim']:.3f}")
print(f"대조군 yes 차이 {R['control_yes_diff']:+.3f} [{R['control_yes_ci'][0]:+.3f}, {R['control_yes_ci'][1]:+.3f}] | 환각 차이 {R['control_hal_diff']:+.3f}")
print(f"\n=== T1b-0 유효성·대조군 안정(|차이| ≤ 0.05) → {'충족' if R['T1b_0'] else '미달(판정 보류)'} ===")
print(f"=== T1b-1 절감: 입력 토큰 감소 중앙값 {R['input_reduction_median']:.3f} (≥ 0.40) → {'충족' if R['T1b_1'] else '미달'} ===")
print(f"=== T1b-2 품질(비열등): yes trim − full {R['yes_diff']:+.3f} [{R['yes_diff_ci'][0]:+.3f}, {R['yes_diff_ci'][1]:+.3f}] (≥ −0.05, 하한 ≥ −0.10) → {'충족' if R['T1b_2'] else '미달'} ===")
print(f"=== T1b-3 충실성: 환각 trim − full {R['hal_diff']:+.3f} [{R['hal_diff_ci'][0]:+.3f}, {R['hal_diff_ci'][1]:+.3f}] (≤ +0.03, 상한 < +0.08) → {'충족' if R['T1b_3'] else '미달'} ===")
print(f"=== T1b {'채택' if R['T1b'] else '미채택'} ===")
print('(보고) partial', R['partial'], '| no', R['no'], '| 잘림', R['trunc'], '| 환각', R['hal'])

## 7. 저장

In [ ]:
usage = {f: round(cost(load_json(os.path.join(T1B_DIR, f))), 2) for f in ['gen.json', 'grade.json', 'faith.json'] if os.path.exists(os.path.join(T1B_DIR, f))}
print('API 사용량(배치 단가) $', usage, '합계', round(sum(usage.values()), 2))
LOCAL = '/content/t1b_local'; os.makedirs(LOCAL, exist_ok=True)
D.to_csv(os.path.join(LOCAL, 't1b_rows.csv'), index=False)           # 원문 없음
json.dump(dict(R, cost_usd=usage, use_temperature=USE_TEMP, n_pool=len(POOL)), open(os.path.join(LOCAL, 't1b_summary.json'), 'w'),
          ensure_ascii=False, indent=1, default=float)
for f in os.listdir(LOCAL): shutil.copy(os.path.join(LOCAL, f), os.path.join(T1B_DIR, f))
zp = shutil.make_archive('/content/t1b_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)